<a href="https://colab.research.google.com/github/emmanguyen01-spec/Coding-Exercise---ML-Basics/blob/main/ML/customer_churn_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import pandas as pd
import numpy as np
import os
import kagglehub

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Data Source:
# Kaggle - Telco Customer Churn by BlastChar
# https://www.kaggle.com/datasets/blastchar/telco-customer-churn

# Download latest version
path = kagglehub.dataset_download(
    "blastchar/telco-customer-churn"
)

print("Path to dataset files:", path)

# Show files in the dataset
print(os.listdir(path))

df = pd.read_csv(
    os.path.join(path, "WA_Fn-UseC_-Telco-Customer-Churn.csv")
)

# Preview the dataset
print(df.head())

print("\nDataset shape:")
print(df.shape)

print("\nColumn names:")
print(df.columns.tolist())

# Keep the columns needed for the assignment
df = df[
    [
        'tenure',
        'MonthlyCharges',
        'TotalCharges',
        'Contract',
        'InternetService',
        'TechSupport',
        'Churn'
    ]
]

# Convert TotalCharges to numeric
# Any blank/non-numeric values will become NaN
df['TotalCharges'] = pd.to_numeric(
    df['TotalCharges'],
    errors='coerce'
)

# Remove rows with missing values
df = df.dropna()

# Convert Churn from Yes/No to 1/0
df['Churn'] = df['Churn'].map({
    'Yes': 1,
    'No': 0
})

# Preview cleaned data
print("\nCleaned Dataset:")
print(df.head())

print("\nNumber of records:")
print(len(df))

# Features and target
X = df[['tenure', 'MonthlyCharges', 'TotalCharges',
        'Contract', 'InternetService', 'TechSupport']]
y = df['Churn']

# Preprocessing: Scale numerical features and one-hot encode categorical features
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['tenure', 'MonthlyCharges', 'TotalCharges']),
        ('cat', OneHotEncoder(sparse_output=False),
         ['Contract', 'InternetService', 'TechSupport'])
    ])

# Create pipeline with preprocessing and model
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(random_state=42))
])

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Train model
model.fit(X_train, y_train)

# Predict churn probability for a new customer
new_customer = pd.DataFrame({
    'tenure': [12],
    'MonthlyCharges': [75.00],
    'TotalCharges': [900.00],
    'Contract': ['Month-to-month'],
    'InternetService': ['Fiber optic'],
    'TechSupport': ['No']
})

churn_probability = model.predict_proba(new_customer)[0][1]
# Probability of churn (class 1)

# Classify based on threshold (0.5)
threshold = 0.5
churn_prediction = 1 if churn_probability > threshold else 0

print(f"\nChurn Probability for new customer: {churn_probability:.2f}")
print(f"Churn Prediction (1 = churn, 0 = no churn): {churn_prediction}")

# Display model coefficients
feature_names = (
    ['tenure', 'MonthlyCharges', 'TotalCharges'] +
    model.named_steps['preprocessor']
    .named_transformers_['cat']
    .get_feature_names_out(
        ['Contract', 'InternetService', 'TechSupport']
    ).tolist()
)

coefficients = model.named_steps['classifier'].coef_[0]

print("\nModel Coefficients:")
for feature, coef in zip(feature_names, coefficients):
    print(f"{feature}: {coef:.2f}")

Using Colab cache for faster access to the 'telco-customer-churn' dataset.
Path to dataset files: /kaggle/input/telco-customer-churn
['WA_Fn-UseC_-Telco-Customer-Churn.csv']
   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL             No  ...               No   
1                No             DSL            Yes  ...              Yes   
2                No             DSL            Yes  ...               No   
3  No phone service             DSL      